# MBA03 · Week 4 — Money has a date on it
**European School of Economics · Thu 15 Oct 2026 · Tutor: Niccolò Salvini**

The case: a new assembly line costs **€2.4M today** and returns **€700k, €800k, €900k, €1,000k** over four years. Arno Bikes borrows at **9%**.

In [1]:
import subprocess, sys
try:
    import plotly
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "plotly"])
import numpy as np, pandas as pd, plotly.graph_objects as go
from math import exp
print("ready")

ready


## A. Compounding, and its ceiling

€1,000 at 12% for one year, compounded `n` times a year.

In [2]:
for label, n in [("once a year", 1), ("every 6 months", 2), ("every quarter", 4),
                 ("every month", 12), ("every week", 52), ("every day", 365),
                 ("every hour", 8760)]:
    print(f"{label:<16} n = {n:>5}   €{1000*(1 + 0.12/n)**n:,.4f}")
print(f"{'continuously':<16} n → ∞     €{1000*exp(0.12):,.4f}   ← the ceiling, and that is e")

once a year      n =     1   €1,120.0000
every 6 months   n =     2   €1,123.6000
every quarter    n =     4   €1,125.5088
every month      n =    12   €1,126.8250
every week       n =    52   €1,127.3410
every day        n =   365   €1,127.4746
every hour       n =  8760   €1,127.4959
continuously     n → ∞     €1,127.4969   ← the ceiling, and that is e


### 🔍 CHECK
The gap from yearly to daily is €7.47. The gap from daily to *infinitely often* is three cents. Say out loud what that means for a bank's marketing.

## B. The effective annual rate — the only honest comparison

In [3]:
def ear(nominal, m):
    return exp(nominal) - 1 if m is None else (1 + nominal/m)**m - 1

print("12% nominal:")
for lab, m in [("semi-annually", 2), ("quarterly", 4), ("monthly", 12), ("continuously", None)]:
    print(f"  compounded {lab:<14} EAR = {ear(0.12, m):.4%}")

print()
print("Bank A: 7.8% monthly  → EAR", f"{ear(0.078, 12):.4%}")
print("Bank B: 8.0% annually → EAR", f"{ear(0.08, 1):.4%}")
print("The lower headline rate is the dearer loan.")

12% nominal:
  compounded semi-annually  EAR = 12.3600%
  compounded quarterly      EAR = 12.5509%
  compounded monthly        EAR = 12.6825%
  compounded continuously   EAR = 12.7497%

Bank A: 7.8% monthly  → EAR 8.0850%
Bank B: 8.0% annually → EAR 8.0000%
The lower headline rate is the dearer loan.


## C. Discounting — what distant money is worth

In [4]:
years = np.arange(0, 21)
fig = go.Figure()
for r, col in ((0.03, "#CDBA80"), (0.09, "#363636"), (0.16, "#AF1F25")):
    fig.add_scatter(x=years, y=1/(1 + r)**years, name=f"{r:.0%}",
                    line=dict(color=col, width=3))
fig.update_layout(template="simple_white", height=420, xaxis_title="years away",
                  yaxis_title="worth of €1 today", legend_title="discount rate")
fig.show()

In [5]:
annuity    = lambda c, r, n: c*(1 - (1 + r)**-n)/r
perpetuity = lambda c, r: c/r
print(f"€10,000 in 5 years at 7%        : €{10000/1.07**5:,.2f}")
print(f"€12,000 a year for 10 yrs at 6% : €{annuity(12000, 0.06, 10):,.2f}")
print(f"€12,000 a year for ever at 6%   : €{perpetuity(12000, 0.06):,.2f}")
print(f"  everything after year 10      : €{perpetuity(12000, 0.06) - annuity(12000, 0.06, 10):,.2f}")

€10,000 in 5 years at 7%        : €7,129.86
€12,000 a year for 10 yrs at 6% : €88,321.04
€12,000 a year for ever at 6%   : €200,000.00
  everything after year 10      : €111,678.96


### What is Arno Bikes worth? — slide the growth rate

Next year's free cash flow €300k, cost of capital 9%, growth `g` for ever: $V = \dfrac{CF_1}{r - g}$.
Move the slider towards 9% and watch the value run up the wall.

In [6]:
CF1, R = 300, 0.09                          # € thousand, cost of capital
gs = np.linspace(0, 0.085, 400)
fig = go.Figure()
fig.add_scatter(x=gs*100, y=CF1/(R - gs)/1000, line=dict(color="#363636", width=3), name="value")
steps = []
for g in np.round(np.arange(0, 0.0801, 0.005), 3):
    fig.add_scatter(x=[g*100], y=[CF1/(R - g)/1000], mode="markers+text", visible=False,
                    marker=dict(size=14, color="#AF1F25"), textposition="middle left",
                    text=[f"g = {g:.1%}: €{CF1/(R - g)/1000:,.2f}M  "], showlegend=False)
n = len(fig.data) - 1
for k in range(n):
    vis = [True] + [j == k for j in range(n)]
    steps.append(dict(method="update", args=[{"visible": vis}], label=f"{fig.data[k+1].x[0]:.1f}%"))
fig.data[1 + 6].visible = True              # start at g = 3%
fig.add_vline(x=R*100, line=dict(color="#AF1F25", dash="dash"), annotation_text="g = r = 9%")
fig.update_layout(template="simple_white", height=460, xaxis_title="growth for ever, g %",
                  yaxis_title="value today, € million", xaxis_range=[0, 10], yaxis_range=[0, 65],
                  sliders=[dict(active=6, steps=steps, currentvalue=dict(prefix="g = "))])
fig.show()
for g in (0.02, 0.03, 0.04):
    print(f"g = {g:.0%}: €{CF1/(R - g):,.2f}k")

g = 2%: €4,285.71k
g = 3%: €5,000.00k
g = 4%: €6,000.00k


### 🔍 CHECK
Going from 3% to 4% adds €1M. Going from 7% to 8% adds how much? Guess, then compute `300/0.01 − 300/0.02`. Why can no firm grow at 9% for ever in this formula?

## D. NPV, and the rate that makes it zero

Move the discount rate. Watch the sign change.

In [7]:
FLOWS = [-2400, 700, 800, 900, 1000]          # € thousand
npv = lambda r: sum(cf/(1 + r)**t for t, cf in enumerate(FLOWS))

print(f"{'year':>5} {'flow':>8} {'factor':>10} {'PV @9%':>10}")
for t, cf in enumerate(FLOWS):
    print(f"{t:>5} {cf:>8,} {1/1.09**t:>10.6f} {cf/1.09**t:>10.2f}")
print(f"{'':>5} {'':>8} {'NPV':>10} {npv(0.09):>10.2f}")

 year     flow     factor     PV @9%
    0   -2,400   1.000000   -2400.00
    1      700   0.917431     642.20
    2      800   0.841680     673.34
    3      900   0.772183     694.97
    4    1,000   0.708425     708.43
                      NPV     318.94


In [8]:
rates = np.linspace(0.001, 0.30, 300)
lo, hi = 0.0, 0.5
for _ in range(80):
    mid = (lo + hi)/2
    lo, hi = (mid, hi) if npv(mid) > 0 else (lo, mid)
irr = (lo + hi)/2

fig = go.Figure(go.Scatter(x=rates*100, y=[npv(r) for r in rates],
                           line=dict(color="#363636", width=3), name="NPV"))
fig.add_hline(y=0, line=dict(color="#7a7f85", width=1))
fig.add_scatter(x=[9], y=[npv(0.09)], mode="markers+text", marker=dict(size=13, color="#1E8449"),
                text=[f"our 9%: +{npv(0.09):,.0f}k"], textposition="top right", showlegend=False)
fig.add_scatter(x=[irr*100], y=[0], mode="markers+text", marker=dict(size=13, color="#CDBA80"),
                text=[f"IRR = {irr:.2%}"], textposition="bottom right", showlegend=False)
fig.update_layout(template="simple_white", height=440, xaxis_title="discount rate %",
                  yaxis_title="NPV (€ thousand)")
fig.show()
print(f"IRR = {irr:.4%}    NPV at 16% = {npv(0.16):,.2f}k")

IRR = 14.5562%    NPV at 16% = -73.14k


### 🔍 CHECK
Delay the first €700k by one year (so the flows become `−2400, 0, 700, 800, 900, 1000`). Before running it: does the IRR rise or fall? Then change `FLOWS` above and find out.

## Drills, solved in code

This section is an optional companion: the code is not examined, but it is a good way to check your work. Every drill from `drills.md` is solved here in plain Python, with the same numbers. Do the drill on paper first, then run the cell and compare.

**D1 (a).** €8,000 is invested for 3 years at 6% a year. Simple interest: the final amount.

$$A = P(1 + r\,t)$$

In [9]:
P = 8000     # principal, €
r = 0.06     # yearly rate
t = 3        # years

A_simple = P * (1 + r * t)    # interest only on the original €8,000
print(f"With simple interest the final amount is €{A_simple:,.2f}.")
assert abs(A_simple - 9440.00) < 0.01

With simple interest the final amount is €9,440.00.


**D1 (b).** Same money, compounded annually: the final amount.

$$A = P(1 + r)^t$$

In [10]:
P, r, t = 8000, 0.06, 3

A_compound = P * (1 + r) ** t    # interest also earns interest
print(f"Compounded annually the final amount is €{A_compound:,.2f}.")
assert abs(A_compound - 9528.13) < 0.01

Compounded annually the final amount is €9,528.13.


**D1 (c).** The difference, and one sentence on where it comes from.

In [11]:
P, r, t = 8000, 0.06, 3
A_simple = P * (1 + r * t)
A_compound = P * (1 + r) ** t

difference = A_compound - A_simple
print(f"Compounding earns €{difference:,.2f} more: interest is paid on the interest already earned.")
assert abs(difference - 88.13) < 0.01

Compounding earns €88.13 more: interest is paid on the interest already earned.


**D2 (a).** €5,000 for 4 years at 8% nominal, compounded quarterly. The final amount.

$$A = P\left(1 + \frac{i}{m}\right)^{m t}$$

In [12]:
P = 5000     # principal, €
i = 0.08     # nominal yearly rate
m = 4        # compounding periods per year (quarterly)
t = 4        # years

A = P * (1 + i / m) ** (m * t)    # 2% a quarter, 16 quarters
print(f"The final amount is €{A:,.2f}.")
assert abs(A - 6863.93) < 0.01

The final amount is €6,863.93.


**D2 (b).** The interest earned.

In [13]:
P, i, m, t = 5000, 0.08, 4, 4
A = P * (1 + i / m) ** (m * t)

interest = A - P
print(f"The interest earned is €{interest:,.2f}.")
assert abs(interest - 1863.93) < 0.01

The interest earned is €1,863.93.


**D3.** A bank quotes 6% nominal. Which is worth more to a depositor?
(i) compounded annually (ii) compounded monthly (iii) they are the same

In [14]:
i = 0.06
ear_annual = (1 + i / 1) ** 1 - 1
ear_monthly = (1 + i / 12) ** 12 - 1

print(f"Annually: EAR = {ear_annual:.2%}.  Monthly: EAR = {ear_monthly:.2%}.")
print("Answer (ii): monthly compounding pays interest on interest sooner, so it is worth more.")
assert ear_monthly > ear_annual
assert abs(ear_monthly - 0.0617) < 0.00005

Annually: EAR = 6.00%.  Monthly: EAR = 6.17%.
Answer (ii): monthly compounding pays interest on interest sooner, so it is worth more.


**D4.** Compute the EAR for a 12% nominal rate compounded (a) semi-annually, (b) quarterly, (c) monthly.

$$\text{EAR} = \left(1 + \frac{i}{m}\right)^m - 1$$

In [15]:
i = 0.12
ear_a = (1 + i / 2) ** 2 - 1      # (a) semi-annually
ear_b = (1 + i / 4) ** 4 - 1      # (b) quarterly
ear_c = (1 + i / 12) ** 12 - 1    # (c) monthly

print(f"(a) semi-annually: EAR = {ear_a:.2%}")
print(f"(b) quarterly:     EAR = {ear_b:.2%}")
print(f"(c) monthly:       EAR = {ear_c:.2%}")
assert abs(ear_a - 0.1236) < 0.00005 and abs(ear_b - 0.1255) < 0.00005
assert abs(ear_c - 0.1268) < 0.00005

(a) semi-annually: EAR = 12.36%
(b) quarterly:     EAR = 12.55%
(c) monthly:       EAR = 12.68%


**D4 (d).** The same 12% compounded continuously.

$$\text{EAR} = e^{i} - 1$$

In [16]:
i = 0.12
ear_d = exp(i) - 1    # the ceiling that (a), (b), (c) approach

print(f"(d) continuously:  EAR = {ear_d:.2%} — the gaps shrink towards this ceiling.")
assert abs(ear_d - 0.1275) < 0.00005

(d) continuously:  EAR = 12.75% — the gaps shrink towards this ceiling.


**D5.** Bank A offers 7.8% compounded monthly. Bank B offers 8% compounded annually. Which do you borrow from, and why?

In [17]:
ear_A = (1 + 0.078 / 12) ** 12 - 1    # put Bank A on an annual basis
ear_B = (1 + 0.08 / 1) ** 1 - 1       # Bank B is already annual

print(f"Bank A: EAR = {ear_A:.3%}.  Bank B: EAR = {ear_B:.3%}.")
print("Borrow from B: A's lower quoted rate hides more frequent compounding.")
assert abs(ear_A - 0.08085) < 0.000005
assert ear_B < ear_A

Bank A: EAR = 8.085%.  Bank B: EAR = 8.000%.
Borrow from B: A's lower quoted rate hides more frequent compounding.


**D6 (stretch).** 5% nominal compounded continuously. What annual compounding rate leaves a depositor equally well off?

In [18]:
i = 0.05
equivalent_annual = exp(i) - 1    # one year of continuous growth, as a yearly rate

print(f"An annual rate of {equivalent_annual:.2%} is equivalent.")
assert abs(equivalent_annual - 0.0513) < 0.00005

An annual rate of 5.13% is equivalent.


**D7.** What is €10,000 receivable in 5 years worth today at 7%?

$$PV = \frac{FV}{(1 + r)^n}$$

In [19]:
FV = 10000   # € received in the future
r = 0.07
n = 5        # years away

PV = FV / (1 + r) ** n
print(f"€10,000 in 5 years is worth €{PV:,.2f} today.")
assert abs(PV - 7129.86) < 0.01

€10,000 in 5 years is worth €7,129.86 today.


**D8.** Pay for a €50,000 machine with (i) €50,000 today or (ii) €18,000 at the end of each of the next three years. Cost of capital 9%. Which do you choose?

$$PV_{\text{annuity}} = C \cdot \frac{1 - (1 + r)^{-n}}{r}$$

In [20]:
C, r, n = 18000, 0.09, 3
price_today = 50000

factor = (1 - (1 + r) ** -n) / r     # annuity factor
PV_instalments = C * factor
print(f"Annuity factor {factor:.6f}; PV of the instalments = €{PV_instalments:,.2f}.")
print(f"Pay in instalments: about €{price_today - PV_instalments:,.0f} less in today's money.")
assert abs(PV_instalments - 45563.31) < 0.02

Annuity factor 2.531295; PV of the instalments = €45,563.30.
Pay in instalments: about €4,437 less in today's money.


**D9 (a).** An annuity pays €12,000 at the end of each year for 10 years. At 6%, what is it worth today?

In [21]:
C, r, n = 12000, 0.06, 10

PV_annuity = C * (1 - (1 + r) ** -n) / r
print(f"The 10-year annuity is worth €{PV_annuity:,.2f} today.")
assert abs(PV_annuity - 88321.04) < 0.01

The 10-year annuity is worth €88,321.04 today.


**D9 (b).** A perpetuity pays €12,000 at the end of every year for ever. At 6%, what is it worth today?

$$PV = \frac{C}{r}$$

In [22]:
C, r = 12000, 0.06

PV_perpetuity = C / r
print(f"The perpetuity is worth €{PV_perpetuity:,.2f} today.")
assert abs(PV_perpetuity - 200000.00) < 0.01

The perpetuity is worth €200,000.00 today.


**D9 (c).** Explain the gap between (a) and (b).

In [23]:
C, r, n = 12000, 0.06, 10
PV_annuity = C * (1 - (1 + r) ** -n) / r
PV_perpetuity = C / r

gap = PV_perpetuity - PV_annuity    # value today of every payment after year 10
print(f"All the payments after year 10 are worth only €{gap:,.0f} today:")
print("discounting shrinks distant money towards nothing.")
assert abs(gap - 111679) < 1

All the payments after year 10 are worth only €111,679 today:
discounting shrinks distant money towards nothing.


**D10 (core).** A perpetuity pays €5,000 next year and grows at 2% a year for ever. At 7%, what is it worth today?

$$PV = \frac{C_1}{r - g}$$

In [24]:
C1 = 5000    # first payment, next year
r = 0.07     # discount rate
g = 0.02     # growth rate of the payment

PV = C1 / (r - g)
print(f"The growing perpetuity is worth €{PV:,.2f} today (€{C1/r:,.2f} with no growth).")
assert abs(PV - 100000.00) < 0.01

The growing perpetuity is worth €100,000.00 today (€71,428.57 with no growth).


**D11 (a).** What is Arno Bikes worth? Free cash flow €350k next year, growing 4% a year for ever, cost of capital 9%.

$$V = \frac{CF_1}{r - g}$$

In [25]:
CF1 = 350    # € thousand, next year
r, g = 0.09, 0.04

V = CF1 / (r - g)
print(f"Arno Bikes is worth €{V:,.2f}k today.")
assert abs(V - 7000.00) < 0.01

Arno Bikes is worth €7,000.00k today.


**D11 (b).** By how much does the value change if growth is one point lower (3% instead of 4%)?

In [26]:
CF1, r = 350, 0.09
V4 = CF1 / (r - 0.04)
V3 = CF1 / (r - 0.03)

change = V3 - V4
print(f"At 3% the value is €{V3:,.2f}k: a change of €{change:,.2f}k, or {change/V4:.2%}.")
assert abs(V3 - 5833.33) < 0.01 and abs(change - (-1166.67)) < 0.01 and abs(change/V4 + 0.1667) < 0.00005

At 3% the value is €5,833.33k: a change of €-1,166.67k, or -16.67%.


**D11 (c).** One sentence for the Board.

In [27]:
CF1, r = 350, 0.09
V4, V3 = CF1 / (r - 0.04), CF1 / (r - 0.03)
print(f"On our forecast Arno Bikes is worth about €{V4/1000:.1f}M; if growth is one point lower it is worth "
      f"€{V3/1000:.1f}M, so the price should rest on the growth we can defend, not the one we hope for.")
assert round(V4/1000, 1) == 7.0 and round(V3/1000, 1) == 5.8

On our forecast Arno Bikes is worth about €7.0M; if growth is one point lower it is worth €5.8M, so the price should rest on the growth we can defend, not the one we hope for.


**D12 (stretch).** A listed rival is valued at €12M; next year's free cash flow is €600k; cost of capital 9%. What growth for ever is the market pricing in?

$$P = \frac{CF_1}{r - g} \quad\Longrightarrow\quad g = r - \frac{CF_1}{P}$$

In [28]:
P = 12000    # € thousand, market value
CF1 = 600    # € thousand, next year
r = 0.09

g_implied = r - CF1 / P
print(f"The market is pricing in growth of {g_implied:.2%} a year for ever.")
assert abs(g_implied - 0.04) < 1e-9
assert abs(CF1 / (r - g_implied) - P) < 1e-6    # plug it back: the formula returns the price

The market is pricing in growth of 4.00% a year for ever.


**D13 (a).** Arno Bikes' assembly line at a 9% cost of capital: write the NPV as a sum (€ thousand).

$$NPV = -2400 + \frac{700}{1.09} + \frac{800}{1.09^2} + \frac{900}{1.09^3} + \frac{1000}{1.09^4}$$

In [29]:
flows = [-2400, 700, 800, 900, 1000]    # € thousand, years 0..4
r = 0.09

terms = [f"{cf}/(1.09)^{t}" for t, cf in enumerate(flows)]
print("NPV = " + " + ".join(terms))

NPV = -2400/(1.09)^0 + 700/(1.09)^1 + 800/(1.09)^2 + 900/(1.09)^3 + 1000/(1.09)^4


**D13 (b).** Compute each discounted flow and the NPV.

In [30]:
flows = [-2400, 700, 800, 900, 1000]    # € thousand
r = 0.09

table = pd.DataFrame({"Year": range(len(flows)), "Flow (€k)": flows})
table["Factor"] = 1 / (1 + r) ** table["Year"]
table["Present value (€k)"] = table["Flow (€k)"] * table["Factor"]
print(table.round({"Factor": 6, "Present value (€k)": 2}).to_string(index=False))

NPV = table["Present value (€k)"].sum()
print(f"NPV = €{NPV:,.2f}k")
assert abs(NPV - 318.94) < 0.01

 Year  Flow (€k)   Factor  Present value (€k)
    0      -2400 1.000000            -2400.00
    1        700 0.917431              642.20
    2        800 0.841680              673.34
    3        900 0.772183              694.97
    4       1000 0.708425              708.43
NPV = €318.94k


**D13 (c).** Accept or reject, and say the sentence you would tell the Board.

In [31]:
flows = [-2400, 700, 800, 900, 1000]
r = 0.09
NPV = sum(cf / (1 + r) ** t for t, cf in enumerate(flows))

decision = "Accept" if NPV > 0 else "Reject"
print(f"{decision}. At our 9% cost of capital the line adds about €{round(NPV)*1000:,.0f}")
print("of value in today's money.")
assert decision == "Accept"

Accept. At our 9% cost of capital the line adds about €319,000
of value in today's money.


**D13b (a).** 60% equity, 40% debt. Shareholders expect 12%, the bank charges 6%, tax rate 25%. Compute the WACC.

$$\text{WACC} = w_e\,r_e + w_d\,r_d\,(1 - T)$$

In [32]:
w_e, w_d = 0.60, 0.40    # weights of equity and debt
r_e, r_d = 0.12, 0.06    # cost of equity, cost of debt
T = 0.25                 # corporate tax rate

WACC = w_e * r_e + w_d * r_d * (1 - T)
print(f"WACC = {w_e*r_e:.1%} + {w_d*r_d*(1-T):.1%} = {WACC:.1%} — the 9% we were handed.")
assert abs(WACC - 0.09) < 1e-9

WACC = 7.2% + 1.8% = 9.0% — the 9% we were handed.


**D13b (b).** Why does the tax rate appear at all, and only on the debt side?

In [33]:
r_d, T = 0.06, 0.25

after_tax_debt = r_d * (1 - T)    # interest is deductible from taxable profit
print(f"Interest is tax-deductible, so a 6% loan really costs {after_tax_debt:.1%} after tax.")
print("Dividends are not deductible, so the equity side gets no (1 - T).")
assert abs(after_tax_debt - 0.045) < 1e-9

Interest is tax-deductible, so a 6% loan really costs 4.5% after tax.
Dividends are not deductible, so the equity side gets no (1 - T).


**D13b (c).** If the mix became 40/60 (more debt), what happens to the WACC and to the NPV of the line?

In [34]:
r_e, r_d, T = 0.12, 0.06, 0.25
flows = [-2400, 700, 800, 900, 1000]
npv = lambda r: sum(cf / (1 + r) ** t for t, cf in enumerate(flows))

WACC_new = 0.40 * r_e + 0.60 * r_d * (1 - T)    # more weight on the cheaper source
print(f"WACC falls from 9.0% to {WACC_new:.1%}; NPV rises from €{npv(0.09):,.2f}k to €{npv(WACC_new):,.2f}k.")
print("Not free: more debt means more risk, and in time r_e and r_d rise with it.")
assert WACC_new < 0.09 and npv(WACC_new) > npv(0.09)

WACC falls from 9.0% to 7.5%; NPV rises from €318.94k to €416.69k.
Not free: more debt means more risk, and in time r_e and r_d rise with it.


**D14 (stretch).** Recompute the NPV at 16%. What does the change in sign tell you, and what is the rate at which it happens?

In [35]:
flows = [-2400, 700, 800, 900, 1000]    # € thousand
npv = lambda r: sum(cf / (1 + r) ** t for t, cf in enumerate(flows))
print(f"NPV at 16% = −€{-npv(0.16):,.2f}k: negative.")

lo, hi = 0.09, 0.16            # NPV > 0 at 9%, < 0 at 16%: the zero is in between
for _ in range(50):            # bisection: halve the interval, keep the sign change
    mid = (lo + hi) / 2
    lo, hi = (mid, hi) if npv(mid) > 0 else (lo, mid)
irr = (lo + hi) / 2
print(f"NPV = 0 at {irr:.1%}: the internal rate of return. Above it the project destroys value.")
assert abs(npv(0.16) - (-73.14)) < 0.01 and abs(irr - 0.146) < 0.0005

NPV at 16% = −€73.14k: negative.
NPV = 0 at 14.6%: the internal rate of return. Above it the project destroys value.


## Homework

`homework.md` — nine drills plus a memo correcting the Board member who added the flows up.
Due Wednesday 21 October, 23:59.